## Neural Sandhi Splitter (PyTorch-Lightning)
**BiLSTM Seq2Seq + Bahdanau Attention**  
Teacher forcing linear annealing (start=0.7 → end=0.3).  

In [1]:
import os
import re
import random
import json
import unicodedata
from pathlib import Path
from typing import List, Tuple, Dict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor

In [2]:
# CONFIG
DATA_XLSX = "/kaggle/input/sandhi-data/sandhi_data.xlsx"
SHEET_NAME = 0
OUT_DIR = Path("pl_runs")
OUT_DIR.mkdir(exist_ok=True)
SEED = 42

# Model / training hyperparams
BATCH_SIZE = 128
EMBED_SIZE = 64
HIDDEN_SIZE = 256
NUM_LAYERS = 1
LR = 1e-3
MAX_EPOCHS = 30
PATIENCE = 5

# Teacher forcing annealing
TF_START = 0.7
TF_END = 0.3
TF_ANNEAL_EPOCHS = MAX_EPOCHS  # linear over full training by default

# Others
DEVICE = "gpu"  # lightning will pick GPU if available
MAX_OUTPUT_LEN = 120
PAD = '<pad>'
SOS = '<sos>'
EOS = '<eos>'
UNK = '<unk>'

pl.seed_everything(SEED, workers=True)

Seed set to 42


42

In [3]:
def normalize_dev(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()
    t = unicodedata.normalize("NFC", t)
    t = re.sub(r'\s+', ' ', t)
    return t

def gold_split_to_marker(gold_split: str) -> str:
    if pd.isna(gold_split):
        return ""
    g = gold_split.strip()
    if '+' in g:
        return g
    tokens = re.split(r'\s+', g)
    return '+'.join(tokens)

In [4]:
df = pd.read_excel(DATA_XLSX, sheet_name=SHEET_NAME, engine="openpyxl")
df['raw'] = df['Word'].map(normalize_dev)
df['gold_split'] = df['Split'].map(normalize_dev).map(gold_split_to_marker)

df = df[df['raw'].str.len() >= 2]
df = df[df['gold_split'].str.len() >= 1]
df = df.drop_duplicates(subset=['raw', 'gold_split']).reset_index(drop=True)
len(df), df.head(3)

(13696,
           Word         Split          raw    gold_split
 0  प्रथमोऽङ्कः  प्रथमः+अङ्कः  प्रथमोऽङ्कः  प्रथमः+अङ्कः
 1      शब्द इव      शब्दः+इव      शब्द इव      शब्दः+इव
 2       इत इतः       इतः+इतः       इत इतः       इतः+इतः)

In [5]:
def length_bucket(l):
    if l <= 4: return 's'
    if l <= 8: return 'm'
    if l <= 12: return 'l'
    return 'xl'

df['bucket'] = df['raw'].apply(len).apply(length_bucket)
train_list, dev_list, test_list = [], [], []

for _, g in df.groupby('bucket'):
    g = g.sample(frac=1, random_state=SEED)
    n = len(g)
    ntrain = int(0.8 * n)
    ndev = int(0.1 * n)
    train_list.append(g.iloc[:ntrain])
    dev_list.append(g.iloc[ntrain:ntrain+ndev])
    test_list.append(g.iloc[ntrain+ndev:])

train_df = pd.concat(train_list).sample(frac=1, random_state=SEED).reset_index(drop=True)
dev_df = pd.concat(dev_list).sample(frac=1, random_state=SEED).reset_index(drop=True)
test_df = pd.concat(test_list).sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Sizes train/dev/test:", len(train_df), len(dev_df), len(test_df))

Sizes train/dev/test: 10956 1367 1373


In [6]:
# Build char-level vocabulary from train set and include '+' and PAD, SOS, EOS
SPECIALS = [PAD, SOS, EOS, UNK, '+']
chars = set()
for s in train_df['raw'].tolist() + train_df['gold_split'].tolist():
    for ch in str(s):
        if ch == ' ':
            continue
        chars.add(ch)
itos = SPECIALS + sorted([c for c in chars if c not in SPECIALS])
stoi = {c:i for i,c in enumerate(itos)}
vocab_size = len(itos)
print("Vocab size:", vocab_size)

Vocab size: 87


In [7]:
class SandhiDataset(Dataset):
    def __init__(self, rows: List[Tuple[str,str]]):
        self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        raw, gold = self.rows[idx]
        return raw, gold

def encode_seq_text(text: str, add_sos_eos=False):
    chars = list(text)
    if add_sos_eos:
        chars = [SOS] + chars + [EOS]
    ids = [stoi.get(ch, stoi[UNK]) for ch in chars]
    return ids

def collate_fn(batch):
    raws, golds = zip(*batch)
    enc_seqs = [encode_seq_text(r, add_sos_eos=False) for r in raws]
    dec_seqs = [encode_seq_text(g, add_sos_eos=True) for g in golds]
    enc_lens = [len(s) for s in enc_seqs]
    dec_lens = [len(s) for s in dec_seqs]
    max_enc = max(enc_lens)
    max_dec = max(dec_lens)
    enc_padded = torch.full((len(batch), max_enc), stoi[PAD], dtype=torch.long)
    dec_padded = torch.full((len(batch), max_dec), stoi[PAD], dtype=torch.long)
    for i, s in enumerate(enc_seqs):
        enc_padded[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    for i, s in enumerate(dec_seqs):
        dec_padded[i, :len(s)] = torch.tensor(s, dtype=torch.long)
    return enc_padded, torch.tensor(enc_lens), dec_padded, torch.tensor(dec_lens), list(raws), list(golds)

train_ds = SandhiDataset(list(train_df[['raw','gold_split']].itertuples(index=False, name=None)))
dev_ds = SandhiDataset(list(dev_df[['raw','gold_split']].itertuples(index=False, name=None)))
test_ds = SandhiDataset(list(test_df[['raw','gold_split']].itertuples(index=False, name=None)))

In [8]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=stoi[PAD])
        self.lstm = nn.LSTM(emb_size, hidden_size, num_layers=num_layers, batch_first=True, bidirectional=True)
    def forward(self, x, lengths):
        emb = self.embedding(x)
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, c) = self.lstm(packed)
        outputs, _ = pad_packed_sequence(packed_out, batch_first=True)
        return outputs, (h, c)

class BahdanauAttention(nn.Module):
    def __init__(self, enc_hidden, dec_hidden):
        super().__init__()
        self.W1 = nn.Linear(enc_hidden, dec_hidden)
        self.W2 = nn.Linear(dec_hidden, dec_hidden)
        self.V = nn.Linear(dec_hidden, 1)
    def forward(self, enc_outputs, dec_hidden, mask=None):
        score = self.V(torch.tanh(self.W1(enc_outputs) + self.W2(dec_hidden).unsqueeze(1))).squeeze(-1)
        if mask is not None:
            score = score.masked_fill(mask == 0, -1e9)
        attn_weights = torch.softmax(score, dim=-1)
        context = torch.bmm(attn_weights.unsqueeze(1), enc_outputs).squeeze(1)
        return context, attn_weights

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, enc_hidden, dec_hidden, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=stoi[PAD])
        self.lstm = nn.LSTM(emb_size + enc_hidden, dec_hidden, num_layers=num_layers, batch_first=True)
        self.attn = BahdanauAttention(enc_hidden, dec_hidden)
        self.fc_out = nn.Linear(dec_hidden + enc_hidden + emb_size, vocab_size)
    def forward_step(self, input_tok, last_hidden, enc_outputs, mask):
        emb = self.embedding(input_tok).unsqueeze(1)
        dec_hidden = last_hidden[0][-1]
        context, attn_weights = self.attn(enc_outputs, dec_hidden, mask)
        lstm_input = torch.cat([emb, context.unsqueeze(1)], dim=-1)
        out, hidden = self.lstm(lstm_input, last_hidden)
        out = out.squeeze(1)
        logits = self.fc_out(torch.cat([out, context, emb.squeeze(1)], dim=-1))
        return logits, hidden, attn_weights

In [9]:
class SandhiLitModule(pl.LightningModule):
    def __init__(self, vocab_size, emb_size, hidden_size, num_layers=1, lr=1e-3,
                 tf_start=0.7, tf_end=0.3, tf_anneal_epochs=30):
        super().__init__()
        self.save_hyperparameters()
        enc_hidden = hidden_size * 2
        self.encoder = Encoder(vocab_size, emb_size, hidden_size, num_layers=num_layers)
        self.decoder = Decoder(vocab_size, emb_size, enc_hidden, hidden_size, num_layers=num_layers)
        self.enc_to_dec_h = nn.Linear(enc_hidden, hidden_size)
        self.enc_to_dec_c = nn.Linear(enc_hidden, hidden_size)
        self.criterion = nn.CrossEntropyLoss(ignore_index=stoi[PAD], reduction='sum')
    def configure_optimizers(self):
        opt = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        return opt

    def forward(self, enc_inputs, enc_lens, dec_targets=None, teacher_forcing_ratio=0.5):
        batch_size = enc_inputs.size(0)
        enc_outputs, (h, c) = self.encoder(enc_inputs, enc_lens)
        mask = (enc_inputs != stoi[PAD]).to(enc_inputs.device)
        h_cat = torch.cat([h[0::2], h[1::2]], dim=-1)
        c_cat = torch.cat([c[0::2], c[1::2]], dim=-1)
        dec_h0 = torch.tanh(self.enc_to_dec_h(h_cat))
        dec_c0 = torch.tanh(self.enc_to_dec_c(c_cat))
        hidden = (dec_h0, dec_c0)
        max_dec = dec_targets.size(1) if dec_targets is not None else MAX_OUTPUT_LEN
        outputs = torch.zeros(batch_size, max_dec, len(itos), device=enc_inputs.device)
        input_tok = torch.full((batch_size,), stoi[SOS], dtype=torch.long, device=enc_inputs.device)
        for t in range(max_dec):
            logits, hidden, _ = self.decoder.forward_step(input_tok, hidden, enc_outputs, mask)
            outputs[:, t, :] = logits
            if dec_targets is not None and torch.rand(1).item() < teacher_forcing_ratio:
                input_tok = dec_targets[:, t]
            else:
                input_tok = logits.argmax(-1)
        return outputs

    def training_step(self, batch, batch_idx):
        enc_padded, enc_lens, dec_padded, dec_lens, raws, golds = batch
        tf = self._current_tf()
        outputs = self(enc_padded, enc_lens, dec_targets=dec_padded, teacher_forcing_ratio=tf)
        loss = self._compute_loss(outputs, dec_padded)
        self.log("train/loss", loss, on_step=True, on_epoch=True, prog_bar=False, batch_size=enc_padded.size(0))
        return loss
        
    # inside SandhiLitModule

    def on_test_start(self) -> None:
        """Prepare buffers to collect predictions/golds for the whole test run."""
        self._test_preds = []
        self._test_golds = []
        self._test_raws = []

    def test_step(self, batch, batch_idx):
        """
        Run one test batch, decode predictions and append to buffers.
        We return nothing special; final aggregation happens in on_test_epoch_end.
        """
        enc_padded, enc_lens, dec_padded, dec_lens, raws, golds = batch
        # ensure on correct device
        enc_padded = enc_padded.to(self.device)
        # produce outputs without teacher forcing
        outputs = self(enc_padded, enc_lens, dec_targets=None, teacher_forcing_ratio=0.0)
        preds = self._decode_logits(outputs)
        # store in-memory for final aggregation
        self._test_preds.extend(preds)
        self._test_golds.extend(golds)
        self._test_raws.extend(raws)

        # optionally log a lightweight batch-level scalar (not required)
        return None

    def on_test_epoch_end(self):
        """
        Called once after all test_step calls are done.
        Compute metrics from the collected buffers and log/save them.
        """
        # make sure buffers exist
        preds = getattr(self, "_test_preds", [])
        golds = getattr(self, "_test_golds", [])
        raws = getattr(self, "_test_raws", [])

        if len(preds) == 0:
            # nothing to do (shouldn't happen)
            return

        # compute metrics (reuse internal helper functions)
        em = self._exact_match(preds, golds)
        prec, rec, f1 = self._char_prf(preds, golds)

        # log metrics so Trainer/test callback prints them
        self.log("test_em", em, prog_bar=True)
        self.log("test_f1", f1, prog_bar=True)

        # also save to disk under default_root_dir (trainer will set it)
        out_dir = Path(self.trainer.log_dir) if (hasattr(self, "trainer") and getattr(self.trainer, "log_dir", None)) else Path(".")
        out_dir = out_dir if out_dir.exists() else Path(".")
        try:
            import pandas as pd
            df = pd.DataFrame({"raw": raws, "gold": golds, "pred": preds})
            df.to_csv(out_dir / "test_predictions.csv", index=False)
            # also save metrics
            import json
            metrics = {"test_em": em, "test_prec": prec, "test_rec": rec, "test_f1": f1}
            with open(out_dir / "test_metrics.json", "w", encoding="utf8") as fh:
                json.dump(metrics, fh, ensure_ascii=False, indent=2)
            print(f"[INFO] Saved test predictions and metrics to {out_dir}")
        except Exception as e:
            # fail silently but print error
            print(f"[WARN] Could not save test outputs: {e}")

    def validation_step(self, batch, batch_idx):
        enc_padded, enc_lens, dec_padded, dec_lens, raws, golds = batch
        outputs = self(enc_padded, enc_lens, dec_targets=None, teacher_forcing_ratio=0.0)
        preds = self._decode_logits(outputs)
        # compute metrics
        em = self._exact_match(preds, golds)
        prec, rec, f1 = self._char_prf(preds, golds)
        self.log("val_em", em, prog_bar=True, batch_size=enc_padded.size(0))
        self.log("val_f1", f1, prog_bar=True, batch_size=enc_padded.size(0))
        return {"preds": preds, "golds": golds}

    def _compute_loss(self, outputs, targets):
        B, T, V = outputs.size()
        loss = self.criterion(outputs.view(B*T, V), targets.view(B*T))
        n_tokens = (targets != stoi[PAD]).sum().item()
        return loss / max(1, n_tokens)

    # metrics & helpers
    def _decode_logits(self, logits):
        preds = logits.argmax(-1).cpu().numpy()
        out = []
        for p in preds:
            out.append(''.join([itos[i] for i in p if itos[i] not in (PAD, SOS, EOS)]))
        return out

    def _exact_match(self, preds, golds):
        return sum(1 for p,g in zip(preds,golds) if p==g)/len(golds)

    def _char_prf(self, preds, golds):
        from collections import Counter
        tp=fp=fn=0
        for p,g in zip(preds,golds):
            cp = Counter(list(p)); cg = Counter(list(g))
            for k in (cp.keys() | cg.keys()):
                a=cp.get(k,0); b=cg.get(k,0)
                tp += min(a,b); fp += max(0, a-b); fn += max(0, b-a)
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        return prec, rec, f1

    def _current_tf(self):
        epoch = self.current_epoch if hasattr(self, "current_epoch") else 0
        s = self.hparams.tf_start; e = self.hparams.tf_end; total = max(1, self.hparams.tf_anneal_epochs)
        frac = min(1.0, epoch / total)
        return float(s + (e - s) * frac)

In [10]:
class SandhiDataModule(pl.LightningDataModule):
    def __init__(self, train_df, dev_df, test_df, batch_size=128):
        super().__init__()
        self.train_df = train_df
        self.dev_df = dev_df
        self.test_df = test_df
        self.batch_size = batch_size
    def setup(self, stage=None):
        self.train_ds = SandhiDataset(list(self.train_df[['raw','gold_split']].itertuples(index=False, name=None)))
        self.dev_ds = SandhiDataset(list(self.dev_df[['raw','gold_split']].itertuples(index=False, name=None)))
        self.test_ds = SandhiDataset(list(self.test_df[['raw','gold_split']].itertuples(index=False, name=None)))
    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, collate_fn=collate_fn, num_workers=2)
    def val_dataloader(self):
        return DataLoader(self.dev_ds, batch_size=self.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)
    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=2)

In [11]:
class MetricPrintCallback(pl.Callback):
    def __init__(self, metric_name="val_em"):
        super().__init__()
        self.metric_name = metric_name
        self.best = None

    def on_validation_end(self, trainer, pl_module):
        metric = trainer.callback_metrics.get(self.metric_name)

        if metric is not None:
            metric = float(metric)

            if self.best is None:
                self.best = metric
                print(f"[INFO] Initial {self.metric_name}: {metric:.4f}")
            else:
                # improvement
                if metric > self.best:
                    print(f"[IMPROVED] {self.metric_name} improved from {self.best:.4f} → {metric:.4f}")
                    self.best = metric
                else:
                    print(f"[INFO] {self.metric_name} = {metric:.4f} (no improvement)")

In [12]:
data_module = SandhiDataModule(train_df, dev_df, test_df, batch_size=BATCH_SIZE)
lit_model = SandhiLitModule(vocab_size=vocab_size, emb_size=EMBED_SIZE, hidden_size=HIDDEN_SIZE,
                            num_layers=NUM_LAYERS, lr=LR,
                            tf_start=TF_START, tf_end=TF_END, tf_anneal_epochs=TF_ANNEAL_EPOCHS)

In [13]:
checkpoint_callback = ModelCheckpoint(
    dirpath=str(OUT_DIR),
    filename="best-{epoch:02d}-{val_em:.4f}",
    monitor="val_em",
    mode="max",
    save_top_k=1,
    verbose=True
)

early_stop = EarlyStopping(
    monitor="val_em",
    mode="max",
    patience=PATIENCE,
    verbose=True
)

metric_printer = MetricPrintCallback(metric_name="val_em")

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator=DEVICE,
    devices=1 if torch.cuda.is_available() else None,
    callbacks=[checkpoint_callback, early_stop, metric_printer],
    default_root_dir=str(OUT_DIR),
    log_every_n_steps=10,
    enable_model_summary=False,
    logger=False,
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [14]:
data_module.setup()
trainer.fit(lit_model, datamodule=data_module)
print("Training finished. Best checkpoint:", checkpoint_callback.best_model_path)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

[INFO] Initial val_em: 0.0000


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Epoch 9, global step 860: 'val_em' was not in top 1


[INFO] val_em = 0.3336 (no improvement)


Validation: |          | 0/? [00:00<?, ?it/s]

Monitored metric val_em did not improve in the last 5 records. Best score: 0.534. Signaling Trainer to stop.
Epoch 10, global step 946: 'val_em' was not in top 1


[INFO] val_em = 0.4236 (no improvement)
Training finished. Best checkpoint: /kaggle/working/pl_runs/best-epoch=05-val_em=0.5340.ckpt


In [15]:
best_ckpt = checkpoint_callback.best_model_path
assert best_ckpt and os.path.exists(best_ckpt), "No checkpoint found."
model_best = SandhiLitModule.load_from_checkpoint(best_ckpt)
model_best.eval()
model_best.freeze()
trainer.test(model_best, datamodule=data_module, verbose=True)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Testing: |          | 0/? [00:00<?, ?it/s]

[INFO] Saved test predictions and metrics to pl_runs


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test_em          │    0.5156591534614563     │
│          test_f1          │    0.7735388278961182     │
└───────────────────────────┴───────────────────────────┘

[{'test_em': 0.5156591534614563, 'test_f1': 0.7735388278961182}]

In [16]:
def predict_batch(model, loader):
    model = model.to(torch.device("cuda" if torch.cuda.is_available() else "cpu"))
    preds_all, golds_all, raws_all = [], [], []
    for enc_padded, enc_lens, dec_padded, dec_lens, raws, golds in tqdm(loader):
        enc_padded = enc_padded.to(model.device)
        with torch.no_grad():
            outputs = model(enc_padded, enc_lens, dec_targets=None, teacher_forcing_ratio=0.0)
        preds = model._decode_logits(outputs)
        preds_all.extend(preds)
        golds_all.extend(golds)
        raws_all.extend(raws)
    return raws_all, golds_all, preds_all

test_loader = data_module.test_dataloader()
raws, golds, preds = predict_batch(model_best, test_loader)
out_df = pd.DataFrame({"raw": raws, "gold": golds, "pred": preds})
out_df.to_csv(OUT_DIR/"test_predictions.csv", index=False)
out_df.head(10)


  0%|          | 0/11 [00:00<?, ?it/s]

,raw,gold,pred
0,आसनं ददाति,आसनम्+ददाति,आसनम्+ददाति
1,स तत्र,सः+तत्र,सः+तत्र
2,दावानल इव,दावानलः+इव,दावानलः+इव
3,कर्मभिर्न,कर्मभिः+न,कर्मभिः+न
4,तण्डुलौदनम्,तण्डुल+ओदनम्,तण्डुलौ+अनम्
5,उभयोरपि,उभयोः+अपि,उभयः+अपि
6,मूर्तिरिव,मूर्तिः+इव,मूर्तिः+इव
7,तवाहिताः,तव+अहिताः,तव+अहिताः
8,यज्ज्ञानं,यत्+ज्ञानं,यज्ज्ञानं
9,प्रतिकृतिरिव,प्रतिकृतिः+इव,प्रतिकृतिः+इव


In [17]:
def exact_match(preds, golds): return sum(1 for p,g in zip(preds,golds) if p==g)/len(golds)
def char_prf(preds, golds):
    from collections import Counter
    tp=fp=fn=0
    for p,g in zip(preds,golds):
        cp=Counter(list(p)); cg=Counter(list(g))
        for k in (cp.keys() | cg.keys()):
            a=cp.get(k,0); b=cg.get(k,0)
            tp += min(a,b); fp += max(0, a-b); fn += max(0, b-a)
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return prec, rec, f1

em = exact_match(preds, golds)
prec, rec, f1 = char_prf(preds, golds)
summary = {"exact_match": em, "char_prec": prec, "char_rec": rec, "char_f1": f1}
with open(OUT_DIR/"metrics.json", "w", encoding="utf8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
summary

{'exact_match': 0.515659140568099,
 'char_prec': 0.6664769925268389,
 'char_rec': 0.9215799412505246,
 'char_f1': 0.7735388002201432}

In [18]:
def predict_word_single(model, word: str):
    enc_ids = encode_seq_text(normalize_dev(word), add_sos_eos=False)
    enc_tensor = torch.tensor([enc_ids], dtype=torch.long).to(model.device)
    enc_lens = torch.tensor([len(enc_ids)], dtype=torch.long).to(model.device)
    with torch.no_grad():
        outputs = model(enc_tensor, enc_lens, dec_targets=None, teacher_forcing_ratio=0.0)
    pred = model._decode_logits(outputs)[0]
    return pred

print("Example:", test_df['raw'].iloc[0], "->", predict_word_single(model_best, test_df['raw'].iloc[0]))

Example: आसनं ददाति -> आसनम्+ददाति


In [19]:
meta = {"stoi": stoi, "itos": itos, "config": {"embed": EMBED_SIZE, "hidden": HIDDEN_SIZE, "batch": BATCH_SIZE, "tf_start": TF_START, "tf_end": TF_END}}
with open(OUT_DIR/"meta.json", "w", encoding="utf8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Saved meta to", OUT_DIR/"meta.json")

Saved meta to pl_runs/meta.json


In [20]:
import shutil
from pathlib import Path
from IPython.display import FileLink

OUTPUT_DIR = Path("pl_runs")
ZIP_NAME = "sandhi_outputs.zip"

if Path(ZIP_NAME).exists():
    Path(ZIP_NAME).unlink()
shutil.make_archive("sandhi_outputs", 'zip', OUTPUT_DIR)

print("ZIP file created:", ZIP_NAME)
FileLink(ZIP_NAME)

ZIP file created: sandhi_outputs.zip


/kaggle/working/sandhi_outputs.zip